# Session 1 — Why did it say that?

**Model Internals Delhi · Week 1 of 8**

---

### Tonight

You'll open a language model, watch it assemble an answer layer by layer, and then
check whether it does that the same way for Hindi as it does for French.

### The stack

`transformers` and `torch`. Nothing else. No interpretability library, nothing to
register for, no remote server. Everything here runs unchanged on your laptop, on
free Colab, or in your work repo on Monday.

### We use hooks from the first cell

HuggingFace has an `output_hidden_states=True` flag that looks like a shortcut.
**We're not using it**, for a reason worth knowing: what it returns changed between
transformers v4 and v5 — which states you get, and whether the last one has already
been normalised. Code written against one version produces a silently wrong plot on
the other.

Under the hood, transformers v5 implements that flag *by registering forward hooks*.
So we'll just register our own. Six lines, works on every version, and you know
exactly what you collected.

| | Checkpoint | Time |
|---|---|---|
| **1** | Write a hook, capture every layer | ~10 min |
| **2** | Decode the captures — the logit lens | ~12 min |
| **3** | Does the model pivot through English? | ~18 min |

### How to not get stuck

Work in **pairs**, swap driver each checkpoint. `# TODO` cells are yours; each has a
`# HINT` under it and a collapsed solution below. Using the solution is fine.
Sitting stuck in silence is not — three-minute rule, raise a hand.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. CPU works, just slower.

---
*Structure adapted from CS 7180: Neural Mechanics (Bau Lab, Northeastern) —*
*`neural-mechanics.baulab.info`. Tooling is plain HuggingFace + PyTorch.*

---
## 1. Setup

Run this now, during introductions. ~60 seconds.

In [ ]:
%pip install -q transformers torch matplotlib

import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)          # inference only tonight
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

NAME = 'gpt2'
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME).to(DEVICE).eval()

print(f'{NAME} on {DEVICE}: {model.config.num_hidden_layers} layers, '
      f'hidden {model.config.hidden_size}')

---
## 2. Does it know anything?

Before looking inside, check there's something worth looking at.

In [ ]:
PROMPT = 'The Eiffel Tower is in the city of'

inputs = tok(PROMPT, return_tensors='pt').to(DEVICE)
logits = model(**inputs).logits

top = logits[0, -1].softmax(-1).topk(5)
for p, t in zip(top.values, top.indices):
    print(f'{tok.decode(t)!r:>12}  {p:.1%}')

> **Mind the space.** `' Paris'` and `'Paris'` are different tokens — the leading
> space is part of the token. This will bite someone tonight.

---
# Checkpoint 1 — Write a hook

A **forward hook** is a function PyTorch calls every time a module finishes running.
It receives the module, its inputs, and its output. Return `None` and you've just
observed. Return a value and you've *replaced* the output.

That's it. That one mechanism is observation, steering, ablation and activation
patching — Weeks 2 and 4 are variations on it.

First, a helper: every architecture names its layer list differently.

In [ ]:
def get_blocks(model):
    """GPT-2 calls it .h, Llama/Qwen/Gemma call it .layers."""
    base = model.base_model              # GPT2Model / LlamaModel / Qwen2Model / ...
    for name in ('h', 'layers', 'blocks'):
        if hasattr(base, name):
            return getattr(base, name)
    raise ValueError(f'layer list not found on {type(base).__name__}')


blocks = get_blocks(model)
print(f'{len(blocks)} blocks, each a {type(blocks[0]).__name__}')

In [ ]:
# The hook pattern. Read it — this is the whole toolkit.

def capture_into(store, key):
    def hook(module, args, output):
        # blocks usually return a tuple (hidden_states, ...); sometimes a bare tensor
        h = output[0] if isinstance(output, tuple) else output
        store[key] = h.detach()
        # returning None == observe only, don't change anything
    return hook


acts = {}
handles = [b.register_forward_hook(capture_into(acts, i)) for i, b in enumerate(blocks)]

model(**inputs)                      # run it — hooks fire during this

for h in handles:
    h.remove()                       # ALWAYS remove your handles

print(f'captured {len(acts)} layers, each {acts[0].shape}')

> **Always `.remove()`.** A forgotten hook silently corrupts every later run in the
> notebook and you will lose an hour to it. Everyone does, once.

In [ ]:
# TODO: pull out the state after layer 6, at the LAST token position.
# HINT: acts[6] is [batch, position, hidden]. You want [0, -1, :].

layer6 = ...
print(layer6.shape)      # want torch.Size([768])

### Say this to your partner before moving on

Those 768 numbers are everything the model has worked out about what comes next, as
of layer 6.

**And here's the trick.** Layer 6's vector lives in the *same space* as layer 12's —
same basis, same units — because every layer *adds* to a shared stream rather than
replacing it. So the model's output head, trained to read the final layer, can be
pointed at layer 6 instead. It'll be wrong, but interestingly wrong.

That's the **logit lens**.

In [ ]:
#@title Checkpoint 1 solution
layer6 = acts[6][0, -1, :]
print(layer6.shape)

---
# Checkpoint 2 — Decode the captures

Two steps: the model's **final norm**, then its **output head**. Another naming
helper, then we're done with plumbing for the night.

Because *we* captured raw block outputs, we know for certain that none of them have
been normalised — so we apply the norm to all of them, uniformly. That certainty is
the whole reason we wrote the hook instead of trusting a flag.

In [ ]:
def get_final_norm(model):
    """GPT-2: ln_f. Llama/Qwen/Gemma: norm. GPT-NeoX: final_layer_norm."""
    base = model.base_model
    for name in ('ln_f', 'norm', 'final_layernorm', 'final_layer_norm'):
        if hasattr(base, name):
            return getattr(base, name)
    raise ValueError(f'final norm not found on {type(base).__name__}')


def logit_lens(prompt):
    """Probabilities over the vocabulary after every layer. Returns [n_layers, vocab]."""
    blocks = get_blocks(model)
    norm   = get_final_norm(model)
    head   = model.get_output_embeddings()     # the unembedding, model-agnostic

    store, handles = {}, []
    for i, b in enumerate(blocks):
        handles.append(b.register_forward_hook(capture_into(store, i)))
    try:
        model(**tok(prompt, return_tensors='pt').to(DEVICE))
    finally:
        for h in handles:
            h.remove()                          # removed even if the run errors

    stacked = torch.stack([store[i][0, -1, :] for i in range(len(blocks))])
    return head(norm(stacked)).softmax(-1)


probs = logit_lens(PROMPT)
print(probs.shape)

In [ ]:
# TODO: print the top 3 predictions at each layer.
# HINT: loop over range(probs.shape[0]); probs[i].topk(3).indices gives token ids;
#       tok.decode(t) turns one back into text.

...

In [ ]:
#@title Checkpoint 2 solution
for i in range(probs.shape[0]):
    words = '  '.join(repr(tok.decode(t)) for t in probs[i].topk(3).indices)
    print(f'layer {i:>2}:  {words}')

### What you should see

Early layers guess something generic. Somewhere in the middle the topic appears.
Late layers converge.

The course's example is *"Miles Davis plays the ___"*, where the prediction walks
**thing → jazz → horn → trumpet**. The answer is assembled across layers, not looked
up in one place.

In [ ]:
def plot_arrival(prompt, answer):
    p = logit_lens(prompt)
    aid = tok(answer, add_special_tokens=False).input_ids[0]
    ranks = (p > p[:, aid].unsqueeze(1)).sum(-1).cpu() + 1

    plt.figure(figsize=(9, 3.5))
    plt.plot(range(len(ranks)), ranks, marker='o')
    plt.yscale('log'); plt.axhline(1, ls=':', c='green')
    plt.xlabel('layer'); plt.ylabel(f'rank of {answer!r}')
    plt.title(prompt); plt.grid(alpha=.3); plt.show()

    hit = [i for i, r in enumerate(ranks) if r == 1]
    print('reaches rank 1 at layer:', hit[0] if hit else 'never')
    return ranks


_ = plot_arrival('The Eiffel Tower is in the city of', ' Paris')
_ = plot_arrival('The Taj Mahal is in the city of', ' Agra')

### Read this carefully

The **layer of arrival** is where the correct answer first hits rank 1 and stays.

If GPT-2 gets the Taj Mahal wrong entirely — final rank isn't 1 — **that is a result,
not a bug.** It's a 124M-parameter model trained on mostly-English web text from
2019. A model that doesn't know a fact tells you nothing about where it stores it.
Keep those cases in a separate bucket from the ones it gets right.

That distinction is most of what separates a trustworthy interpretability result
from a worthless one.

---
## Interlude — the cold open, explained

Remember the model saying Rome? Same hook, one line different: **return a value
instead of `None`** and you've overwritten the layer's output.

Don't worry about mastering this — hooks are Week 2. But it's worth seeing tonight
that observation and intervention are the same mechanism.

In [ ]:
def overwrite_with(layer_idx, donor_prompt):
    """Replace one layer's last-token state with the state from a different prompt."""
    store, hs = {}, []
    blocks = get_blocks(model)
    hs.append(blocks[layer_idx].register_forward_hook(capture_into(store, 'donor')))
    model(**tok(donor_prompt, return_tensors='pt').to(DEVICE))
    for h in hs:
        h.remove()
    donor = store['donor'][0, -1, :]

    def hook(module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        h[0, -1, :] = donor
        return (h,) + output[1:] if isinstance(output, tuple) else h   # <-- the change

    return blocks[layer_idx].register_forward_hook(hook)


handle = overwrite_with(7, 'The Colosseum is in the city of')
out = model(**tok(PROMPT, return_tensors='pt').to(DEVICE)).logits
handle.remove()

print('prompt still says Eiffel Tower. Model now says:',
      repr(tok.decode(out[0, -1].argmax())))

---
# Checkpoint 3 — Does the model think in English?

Wendler et al. (2024) found that models translating between two non-English
languages appear to pivot through **English** in their middle layers.

GPT-2 is English-only, so switch models. **Nothing above changes** — the helpers
look up the right names themselves. That portability is why we wrote them.

In [ ]:
NAME = 'Qwen/Qwen2.5-0.5B'          # open, multilingual, fits a free T4
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME).to(DEVICE).eval()

print(f'{NAME}: {model.config.num_hidden_layers} layers, '
      f'blocks live at .{type(get_blocks(model)).__name__}')

In [ ]:
WENDLER = '''Français: "cinq" - 中文: "五"
Français: "coeur" - 中文: "心"
Français: "trois" - 中文: "三"
Français: "nuage" - 中文:'''

p = logit_lens(WENDLER)
for i in range(p.shape[0]):
    words = '  '.join(repr(tok.decode(t)) for t in p[i].topk(4).indices)
    print(f'layer {i:>2}:  {words}')

Look at the **middle** rows, not the last. English words showing up in a prompt that
contains no English is the Wendler result. A 0.5B model gives a weaker signal than
the Llama-2 in the paper — the effect should still be visible.

### Your turn

**TODO.** Same experiment, Indian language pair. Copy the shape of `WENDLER`: three
complete pairs, then a fourth with the answer missing.

Nobody has published this. Whatever you find is new — including nothing.

In [ ]:
# TODO: a few-shot translation prompt for an Indian language pair.
#
# HINT — Hindi to Tamil:
# MY_PROMPT = '''हिन्दी: "पानी" - தமிழ்: "தண்ணீர்"
# हिन्दी: "आग" - தமிழ்: "நெருப்பு"
# हिन्दी: "किताब" - தமிழ்: "புத்தகம்"
# हिन्दी: "सूरज" - தமிழ்:'''
#
# Also try: Bengali->Hindi, Tamil->Telugu, Hindi->Marathi,
# or Hinglish (Latin script) -> Hindi (Devanagari).

MY_PROMPT = ...

p = logit_lens(MY_PROMPT)
for i in range(p.shape[0]):
    words = '  '.join(repr(tok.decode(t)) for t in p[i].topk(4).indices)
    print(f'layer {i:>2}:  {words}')

### Log it

Shared sheet: `[FACILITATOR: paste link before session]`

| language pair | English mid-stack? | which layers | when does the target script appear | notes |
|---|---|---|---|---|

Negative results go in too — *"no English, went straight to Tamil"* is just as
interesting and possibly more so.

---
## Before you believe any of this

You have a striking observation. **It is not a finding yet.** Three alternatives,
all live:

| Alternative | Why it's plausible |
|---|---|
| **The output head is biased toward English** | It was trained on mostly-English data. Maybe it maps *any* vector onto English tokens — in which case you're reading the decoder, not the model |
| **Tokenization artifact** | Devanagari and Tamil fragment into far more tokens than Latin script. That alone changes the picture |
| **The English states are epiphenomenal** | Nothing here shows the model *uses* them. Could be a byproduct that never touches the output |

**For the debrief:** pick one. Design the experiment that rules it out. What do you
measure? What's the control? What result would actually convince you?

Controls are Week 3. The causal test — reaching in and changing things, like the
Rome demo — is Week 4.

---

## What you can now do at work

- Register a forward hook on any module of any HuggingFace model and capture its output
- Decode an intermediate state to see what the model believed partway through
- Return a value from a hook to overwrite an activation mid-forward-pass

No library, no version pinning, no vendor. That's the foundation for runtime probes,
steering and patching — Weeks 2, 4 and 5.

## Milestone for Week 2

1. **Form a team** — ideally 3, at least one of whom knows a language or a domain
   better than the rest of you.
2. **Bring 2–3 candidate concepts.** You'll pitch one for 4 minutes and the room
   applies **FINER**: Feasible, Interesting, Novel, Ethical, Relevant.
3. Feasible kills most ideas. It means *are there signs of life in the internals?*
   You now have a way to check that before committing three months.